# Decision Tree Regressor Example (Wine Quality Dataset)

Here it is demonstrated how to use the `DecisionTreeRegressor` module from the CMOR-438 library to predict wine quality scores.
In this example, the Wine Quality dataset is used to train, test, and evaluate the model.

**Goal: Predict the continuous quality score of red wine (3–8) based on its physicochemical properties.**

The regression tree partitions the feature space into rectangular regions and predicts the mean target value within each region.

## 1. Setup and Data Loading

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, '../_shared')
from regression_trees import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = pd.read_csv('../../../data/WineQT.csv').drop(columns=['Id'])
FEATURE_COLS = [c for c in wine.columns if c != 'quality']
print(f"Dataset loaded: {wine.shape[0]} samples, {len(FEATURE_COLS)} features.")
print(f"Target — mean: {wine['quality'].mean():.2f}  std: {wine['quality'].std():.2f}")

## 2. Preprocessing

Standardise features and split 80/20. No binning needed — the target stays continuous.

In [ ]:
X = StandardScaler().fit_transform(wine[FEATURE_COLS].values.astype(float))
y = wine['quality'].values.astype(float)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {X_tr.shape[0]}  |  Test samples: {X_te.shape[0]}")

## 3. Train

Train a Regression Tree using variance reduction as the splitting criterion.
`min_samples_leaf=5` prevents very small leaf nodes that would overfit.

In [ ]:
dtr = DecisionTreeRegressor(max_depth=6, min_samples_leaf=5)
dtr.fit(X_tr, y_tr)
print(f'R²:  {dtr.score(X_te, y_te):.4f}')
print(f'MSE: {dtr.mse(X_te, y_te):.4f}')
print(f'Actual tree depth: {dtr.get_depth()}')

## 4. Results and Visualisation

Two plots are produced:
- **Predicted vs actual** — each point is a test sample; perfect predictions lie on the red diagonal line; the spread shows prediction error
- **Residual distribution** — histogram of (actual − predicted); a well-centred distribution around zero indicates unbiased predictions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
preds = dtr.predict(X_te)

axes[0].scatter(y_te, preds, alpha=0.4, s=18, color='teal')
lo, hi = y_te.min()-0.2, y_te.max()+0.2
axes[0].plot([lo,hi],[lo,hi],'r--',lw=1.5,label='Perfect fit')
axes[0].set_xlabel('Actual Quality'); axes[0].set_ylabel('Predicted Quality')
axes[0].set_title(f'Regression Tree - Predicted vs Actual  R²={dtr.score(X_te,y_te):.3f}', fontweight='bold')
axes[0].legend()

residuals = y_te - preds
axes[1].hist(residuals, bins=30, color='darkorchid', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_xlabel('Residual'); axes[1].set_ylabel('Count')
axes[1].set_title('Regression Tree - Residual Distribution', fontweight='bold')
plt.tight_layout(); plt.show()